# Data Analysis
This notebook is used to analyze the data collected from live runs. It includes various visualizations and statistical analyses to understand the performance of the runner

## TOC:
* [Clean Data](#cleaning-data)
* [Performance from Specific Runs](#performance-from-specific-runs)
* [Simulation vs Reality](#simulation-vs-reality)

In [ ]:
from utilis.helper import *
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
from simulation.monte_carlo_simulation import MonteCarloSimulation, create_dataframes
from model_fit import load_run_data
from simulation.data_classes import Params, SimConfig
from simulation.pacing_strategy import ConstantPaceStrategy, EvenEffortStrategy

output_folder = extract_global_json("output_folder")


### Cleaning data
This section is used to clean the data before analysis. It includes removing unnecessary columns, handling missing values, and converting data types.

In [ ]:
# Show all rows
pd.set_option('display.max_rows', None)

# Show all columns
pd.set_option('display.max_columns', None)

# Optional: widen the display to avoid line wrapping
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)  # or a large int, e.g. 200

# folder = "2025-07-05_12-32"
# folder = "2025-07-22_19-40"
# folder = "2025-07-25_19-53"
# folder = "2025-07-27_10-27"
# folder = "2025-07-29_20-03"
# folder = "2025-07-31_20-07"
# folder = "2025-08-02_18-02"
# folder = "2025-08-04_20-02"
folder = "2025-08-06_20-14"
# folder = "2025-10-10_10-42"
# loop through all folders in the output folder
for folder_name in os.listdir(output_folder):
    if folder in folder_name:
        # first read the csv file
        csv_file_path = os.path.join(output_folder, folder_name, f"{folder_name}_streams.csv")
        json_file_path = os.path.join(output_folder, folder_name, f"{folder_name}_overall.json")

        break
# read the csv file
data = pd.read_csv(csv_file_path)
overall_data = extract_json(json_file_path)

data.dropna(inplace=True)

### Performance from Specific Runs
This section will focus on analyzing the performance from specific runs, including visualizations and statistical summaries.

In [ ]:
# y = "pace_efficiency"
# y = "altitude_m"
# y = "smooth_altitude_m"
# y = "stride_length_m"
# y = "heartrate_bpm"
# y = "heartrate_smooth_bpm"
# y = "diff_heartrate_smooth_bpm"
# y = "smooth_headwind_mps"
y = "velocity_mps"
# y = "headwind_mps"
# y = "stride_length_m"
# y = "stride_length_smooth_m"
# y = "diff_heartrate_bpm"
# y = "cadence_rpm"

# x = "cadence_rpm"
# x = "grade_percent"
x = "time_datetime"
# x = "time_s"
# x = "distance_m"

# x_list = []
# y_list = []

# get the time and cadence columns
x_data = data[x]
y_data = data[y]

# plot this data
plt.figure(figsize=(10, 6))
plt.scatter(x_data, y_data, label=folder_name)
# plt.plot(x_data, y_data, label=folder_name)
plt.xlabel(x)
plt.ylabel(y)
plt.title(f"{y} vs {x} for {folder_name}")
plt.show()

### Simulation vs Reality
This section will compare the simulation results with the actual performance data to evaluate the accuracy of the simulation model.

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt

def generate_realistic_noise(length, target_rmse, cutoff_hz=0.1, fs=1.0):
    # 1. Generate raw Gaussian noise
    raw_noise = np.random.normal(0, 1, length)

    # 2. Design a Low-Pass Butterworth Filter
    # nyquist frequency is half the sampling rate
    nyq = 0.5 * fs
    normal_cutoff = cutoff_hz / nyq
    b, a = butter(N=2, Wn=normal_cutoff, btype="low", analog=False)

    # 3. Apply the filter (filtfilt prevents phase shift/delay)
    smooth_noise = filtfilt(b, a, raw_noise)

    # 4. Scale it to match your Optuna Error (0.435 m/s)
    # We scale by the ratio of target_rmse to the current std of the smooth signal
    current_std = np.std(smooth_noise)
    return smooth_noise * (target_rmse / current_std)

# determine which run to use for fitting the model parameters
date = "2025-08-06_20-14"
# date = "2025-10-10_10-42"
csv_data = f"runs/{date}/{date}_streams.csv"
json_data = f"runs/{date}/{date}_overall.json"

# get input dataframe for the simulation
run_data = load_run_data(csv_data, json_data)

# obtained from optimization
pacing = "constant"
# get constant values for the simulation
params = Params(
    F=[11.43553402602856],
    E0=[2258.19432053275],
    tau=[0.9811614674481888],
    sigma=[21.74517480450133],
    gamma=[5.985038605251743e-05],
    drag_coefficient=[1.0585890773336417],
    frontal_area=[0.45878756196457837],
    mass=[79.98506233690507],
    rho=[1.225],
    convection=[10.0],
    alpha=[0.7985623375601472],
    psi=[0.006995627918878894],
)
# create the simulation configuration
sim_cfg = SimConfig(
    target_dist=run_data["total_distance"],
    num_sim=1,
    dt=0.1,
    max_steps=20000,
    const_v=4.3169832045132885,
    t1=None,
    t2=None,
)

df_input = create_dataframes(params, sim_cfg.num_sim)

strat = ConstantPaceStrategy(sim_cfg) if pacing == "constant" else EvenEffortStrategy(sim_cfg)

# perform simulation
sim = MonteCarloSimulation(sim_cfg, strat, df_input=df_input, csv_data=csv_data, json_data=json_data)
sim.run()

# get the velocity and time arrays from the simulation and make it into a pandas dataframe
v_sim = sim.velocity[:, 0]
t_sim = sim.time_elapsed
df_sim = pd.DataFrame({
    "time": t_sim,
    "velocity": v_sim,
})

# get the actual velocity and time arrays from the run data and make it into a pandas dataframe
v_obs = run_data["velocity"]
t_obs = run_data["time"]
df_obs = pd.DataFrame({
    "time": t_obs,
    "velocity": v_obs,
})



# create a mask to filter the simulation dataframe to only include the time points that are present in the observed dataframe
df_sim_masked = pd.DataFrame({
    "time": df_obs["time"].to_numpy(),
    "velocity": np.interp(df_obs["time"], df_sim["time"], df_sim["velocity"]),
})

# add noise to the simulated velocity to make it more realistic
noise = generate_realistic_noise(len(df_sim_masked), target_rmse=(0.1789545301641814)**0.5, cutoff_hz=0.0391, fs=1.0)

df_noise = df_sim_masked.copy()
df_noise["velocity"] += noise


# plot the simulated and observed velocity vs time
plt.figure(figsize=(10, 6))
plt.plot(df_noise["time"], df_noise["velocity"], label="Simulated", color="blue")
plt.scatter(df_obs["time"], df_obs["velocity"], label="Observed", color="orange")
plt.xlabel("Time (s)")
plt.ylabel("Velocity (m/s)")
plt.title(f"Velocity vs Time for {folder_name}")
plt.legend()
plt.show()


In [ ]:
# look at the spectral density
from scipy.signal import welch
import numpy as np
from scipy import signal

def automatic_cutoff(velocity_residuals, fs=1.0, threshold=0.90):
    # 1. Compute PSD
    freqs, psd = signal.welch(velocity_residuals.to_numpy(), fs, nperseg=256)

    # 2. Compute Cumulative Power
    cumulative_psd = np.cumsum(psd)
    total_power = cumulative_psd[-1]

    # 3. Find where the power crosses the threshold (e.g., 90%)
    cutoff_idx = np.where(cumulative_psd >= total_power * threshold)[0][0]
    return freqs[cutoff_idx]

frequencies, power_spectral_density = welch(
    df_obs["velocity"].to_numpy(),
    fs=1.0,
    window="hann",
    nperseg=min(256, len(df_obs["velocity"].to_numpy())),
    noverlap=None,
    detrend="constant",
    scaling="density")

f_c = automatic_cutoff(df_obs["velocity"] - df_sim_masked["velocity"])

print(f"Automatic cutoff frequency: {f_c:.4f} Hz")

plt.figure(figsize=(10, 6))
plt.semilogy(frequencies, power_spectral_density)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Power Spectral Density")
plt.title(f"Power Spectral Density of Velocity for {folder_name}")
plt.show()